In [1]:
"""
OmniOC — Kafka → Azure Event Hubs Mirror Consumer
===================================================
Reads all 4 topics from local Docker Kafka and forwards
every message to the matching Azure Event Hubs topic.

Run in Anaconda Jupyter — one cell at a time.

Install first:
    pip install kafka-python

Topics mirrored:
    clickstream-events  →  eh-clickstream
    device-telemetry    →  device-telemetry
    order-events        →  order-events
    dq-alerts           →  dq-alerts

Consumer groups on Event Hubs (already created):
    cg-dbx-stream   ← used by Databricks streaming notebooks AND this mirror consumer
    cg-debug        ← used for manual spot-checks only
"""

# ═══════════════════════════════════════════════════════════════════════════════
#  CONTROL PANEL — Edit before running
# ═══════════════════════════════════════════════════════════════════════════════

# ── Local Kafka (Docker) ───────────────────────────────────────────────────────
LOCAL_BOOTSTRAP = "localhost:9092"

# ── Azure Event Hubs ───────────────────────────────────────────────────────────
EH_NAMESPACE    = "omnioc-dev-eus-ehns-std-01"
EH_BOOTSTRAP    = f"{EH_NAMESPACE}.servicebus.windows.net:9093"

# Paste your Event Hubs connection string here
# Get it from: Event Hubs → omnioc-dev-eus-ehns-std-01
#              → Shared access policies → RootManageSharedAccessKey
#              → Connection string–primary key
EH_CONNECTION_STRING = (
    "Endpoint=sb://omnioc-dev-eus-ehns-std-01.servicebus.windows.net/;"
    "SharedAccessKeyName=RootManageSharedAccessKey;"
    "SharedAccessKey=NhuUvoHUnk6OpZgg5UJLV9SeewLNZdWRX+AEhP6mU78="
)
# ── Topic Mapping: local Kafka → Event Hubs ────────────────────────────────────
TOPICS_MAP = {
    "clickstream-events" : "eh-clickstream",
    "device-telemetry"   : "device-telemetry",
    "order-events"       : "order-events",
    "dq-alerts"          : "dq-alerts",
}

# ── Consumer behaviour ─────────────────────────────────────────────────────────
CONSUMER_GROUP      = "cg-dbx-stream"        # local Kafka consumer group — matches EH consumer group name for clarity
AUTO_OFFSET_RESET   = "earliest"             # 'earliest' = read ALL messages including old ones
                                             # 'latest'   = read only NEW messages from now
BATCH_SIZE          = 50                     # flush to EH every N messages
IDLE_TIMEOUT_MS     = 5000                   # stop if no messages for 5 seconds (0 = run forever)

# ═══════════════════════════════════════════════════════════════════════════════
#  END CONTROL PANEL
# ═══════════════════════════════════════════════════════════════════════════════



In [2]:
from kafka import KafkaConsumer, KafkaProducer
from kafka.errors import KafkaError
import json, time
from datetime import datetime, timezone


# ─── Step 1: Connect to local Kafka consumer ──────────────────────────────────

def connect_local_consumer():
    print("🔌 Connecting to local Kafka...")
    consumer = KafkaConsumer(
        *TOPICS_MAP.keys(),                          # subscribe to all 4 topics
        bootstrap_servers   = LOCAL_BOOTSTRAP,
        group_id            = CONSUMER_GROUP,
        auto_offset_reset   = AUTO_OFFSET_RESET,
        enable_auto_commit  = True,
        auto_commit_interval_ms = 1000,
        value_deserializer  = lambda v: v,           # keep raw bytes — forward as-is
        key_deserializer    = lambda k: k,           # keep raw bytes
        consumer_timeout_ms = IDLE_TIMEOUT_MS if IDLE_TIMEOUT_MS > 0 else -1,
        fetch_max_wait_ms   = 500,
        max_poll_records    = 100,
    )
    print(f"✅ Local Kafka connected: {LOCAL_BOOTSTRAP}")
    print(f"   Subscribed topics : {list(TOPICS_MAP.keys())}")
    print(f"   Consumer group    : {CONSUMER_GROUP}")
    print(f"   Offset reset      : {AUTO_OFFSET_RESET}")
    return consumer


In [3]:
# ─── Step 2: Connect to Event Hubs producer ───────────────────────────────────

def connect_eh_producer():
    print("\n🔌 Connecting to Azure Event Hubs...")
    producer = KafkaProducer(
        bootstrap_servers   = EH_BOOTSTRAP,
        security_protocol   = "SASL_SSL",
        sasl_mechanism      = "PLAIN",
        sasl_plain_username = "$ConnectionString",
        sasl_plain_password = EH_CONNECTION_STRING,
        value_serializer    = lambda v: v if isinstance(v, bytes) else v,
        key_serializer      = lambda k: k if isinstance(k, bytes) else (k.encode() if k else None),
        request_timeout_ms  = 30000,
        retry_backoff_ms    = 1000,
        max_block_ms        = 60000,
        acks                = 1,                     # EH doesn't support acks='all'
        linger_ms           = 10,                    # small batch for latency
    )
    print(f"✅ Event Hubs connected: {EH_BOOTSTRAP}")
    print(f"   Topics will land in : {list(TOPICS_MAP.values())}")
    return producer


In [4]:
# ─── Step 3: Mirror loop ──────────────────────────────────────────────────────

def mirror_kafka_to_eventhub():
    """
    Main mirror function.
    Reads from local Kafka, forwards to Event Hubs.
    Prints a running count per topic.
    """
    consumer = connect_local_consumer()
    producer = connect_eh_producer()

    # Per-topic counters
    counters = {t: 0 for t in TOPICS_MAP.keys()}
    total    = 0
    errors   = 0

    print("\n" + "═" * 62)
    print("  ⚡ Mirror running — local Kafka → Azure Event Hubs")
    print("  Interrupt kernel (■) to stop")
    print("═" * 62 + "\n")

    start_time = time.time()

    try:
        for msg in consumer:
            local_topic = msg.topic
            eh_topic    = TOPICS_MAP.get(local_topic, local_topic)

            try:
                # Forward message — preserve key and raw value bytes
                future = producer.send(
                    topic = eh_topic,
                    key   = msg.key,
                    value = msg.value,
                )

                counters[local_topic] += 1
                total += 1

                # Flush every BATCH_SIZE messages
                if total % BATCH_SIZE == 0:
                    producer.flush()
                    elapsed = time.time() - start_time
                    rate    = total / elapsed if elapsed > 0 else 0
                    print(f"  📤 Flushed batch | Total: {total:>5} msgs | "
                          f"{rate:.1f} msg/s | Errors: {errors}")
                    for t, c in counters.items():
                        eh = TOPICS_MAP[t]
                        print(f"     {t:<22} → {eh:<20} : {c:>5} msgs")
                    print()

            except KafkaError as e:
                errors += 1
                print(f"  ❌ Failed to send to EH ({eh_topic}): {e}")
                if errors > 10:
                    print("  ⛔ Too many errors — stopping mirror")
                    break

    except StopIteration:
        # consumer_timeout_ms reached — no more messages
        print(f"\n  ⏱️  No messages for {IDLE_TIMEOUT_MS/1000:.0f}s — consumer stopping")

    except KeyboardInterrupt:
        print("\n  🛑 Interrupted by user")

    finally:
        producer.flush()
        producer.close()
        consumer.close()

        # ── Final summary ─────────────────────────────────────────────────
        elapsed = time.time() - start_time
        print("\n" + "═" * 62)
        print("  📊 MIRROR COMPLETE — Final Summary")
        print("═" * 62)
        print(f"  {'Local Kafka Topic':<25} {'Event Hubs Topic':<22} {'Count':>6}")
        print(f"  {'-'*25} {'-'*22} {'-'*6}")
        for t, c in counters.items():
            print(f"  {t:<25} {TOPICS_MAP[t]:<22} {c:>6}")
        print(f"  {'-'*25} {'-'*22} {'-'*6}")
        print(f"  {'TOTAL':<25} {'':<22} {total:>6}")
        print(f"\n  Time elapsed : {elapsed:.1f}s")
        print(f"  Errors       : {errors}")
        print(f"  Avg rate     : {total/elapsed:.1f} msg/s" if elapsed > 0 else "")
        print("═" * 62)
        print("\n  ✅ Check Event Hubs in Azure Portal:")
        print(f"     https://portal.azure.com → {EH_NAMESPACE}")
        print("     → eh-clickstream → Overview → Messages chart")


In [5]:
# ─── Step 4: Verify what landed in Event Hubs ─────────────────────────────────

def verify_eventhub_consumer(topic="eh-clickstream", max_messages=5):
    """
    Quick verification — reads a few messages back from Event Hubs
    using the cg-dbx-stream consumer group — same group Databricks uses.
    NOTE: This will advance the offset for cg-dbx-stream, so run sparingly.
    """
    print(f"\n🔍 Verifying Event Hubs topic: {topic}")
    print(f"   Reading up to {max_messages} messages from cg-dbx-stream group...\n")

    consumer = KafkaConsumer(
        topic,
        bootstrap_servers   = EH_BOOTSTRAP,
        security_protocol   = "SASL_SSL",
        sasl_mechanism      = "PLAIN",
        sasl_plain_username = "$ConnectionString",
        sasl_plain_password = EH_CONNECTION_STRING,
        group_id            = "cg-dbx-stream",       # same CG Databricks uses — verifies exactly what Databricks will read
        auto_offset_reset   = "earliest",
        value_deserializer  = lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms = 5000,
    )

    count = 0
    for msg in consumer:
        if count >= max_messages:
            break
        print(f"  Message #{count + 1}")
        print(f"  Partition : {msg.partition} | Offset: {msg.offset}")
        print(f"  Key       : {msg.key}")
        # Print first few fields of the event
        val = msg.value
        if isinstance(val, dict):
            preview = {k: val[k] for k in list(val.keys())[:6]}
            print(f"  Payload   : {json.dumps(preview, indent=4, default=str)}")
        print()
        count += 1

    consumer.close()
    print(f"✅ Verified {count} messages in {topic}")
    if count == 0:
        print("⚠️  No messages found — run mirror_kafka_to_eventhub() first")



In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ENTRY POINT — run directly
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    mirror_kafka_to_eventhub()


🔌 Connecting to local Kafka...
✅ Local Kafka connected: localhost:9092
   Subscribed topics : ['clickstream-events', 'device-telemetry', 'order-events', 'dq-alerts']
   Consumer group    : eh-mirror-consumer
   Offset reset      : earliest

🔌 Connecting to Azure Event Hubs...
✅ Event Hubs connected: omnioc-dev-eus-ehns-std-01.servicebus.windows.net:9093
   Topics will land in : ['eh-clickstream', 'device-telemetry', 'order-events', 'dq-alerts']

══════════════════════════════════════════════════════════════
  ⚡ Mirror running — local Kafka → Azure Event Hubs
  Interrupt kernel (■) to stop
══════════════════════════════════════════════════════════════

  📤 Flushed batch | Total:    50 msgs | 30.5 msg/s | Errors: 0
     clickstream-events     → eh-clickstream       :    30 msgs
     device-telemetry       → device-telemetry     :     0 msgs
     order-events           → order-events         :    20 msgs
     dq-alerts              → dq-alerts            :     0 msgs

  📤 Flushed batch | 

In [7]:
# from kafka import KafkaProducer
# import json

# EH_CONNECTION_STRING = "Endpoint=sb://omnioc-dev-eus-ehns-std-01.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=NhuUvoHUnk6OpZgg5UJLV9SeewLNZdWRX+AEhP6mU78="

# producer = KafkaProducer(
#     bootstrap_servers   = "omnioc-dev-eus-ehns-std-01.servicebus.windows.net:9093",
#     security_protocol   = "SASL_SSL",
#     sasl_mechanism      = "PLAIN",
#     sasl_plain_username = "$ConnectionString",
#     sasl_plain_password = EH_CONNECTION_STRING,
#     request_timeout_ms  = 60000,
#     max_block_ms        = 120000,
#     acks                = 1,
# )

# # Send one test message
# future = producer.send(
#     "eh-clickstream",
#     value=json.dumps({"test": "connection_check"}).encode()
# )
# result = future.get(timeout=30)
# producer.flush()
# producer.close()

# print(f"✅ Test message sent! Partition={result.partition}, Offset={result.offset}")